# 05 — Statistical Analysis + Scorecard (v3)

Main goals (business pipeline style):
1) Build **video-level metrics** from thread-level sentiment + engagement
2) Estimate uncertainty with **weighted video-block bootstrap**
3) Build a **min-max scorecard** and compare distributions across 2×2 cells

Inputs (Data-only recommended):
- sentiment5_top_level.parquet
- thread_panel_base.parquet

Outputs:
- Data/data_05_scorecard_YYYYMMDD_HHMMSS/
  - video_metrics.csv
  - bootstrap_results.csv
  - scorecard_video.csv
  - scorecard_cell_summary.csv
  - figures/*.png
  - report.json


In [ ]:
# 📌 Cell 1 - Imports
# ------------------------------------------
import os
import glob
import json
import math
from datetime import datetime, timezone
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt


In [ ]:
# 📌 Cell 2 - Locate inputs (Data-only first, fallback to manual path)
# -------------------------------------------------------------------
# ✅ 规则：
# 1) 优先从 Data/ 最新 data_04_sentiment5_* 找 sentiment5_top_level.parquet
# 2) 优先从 Data/ 最新 data_02_* 找 thread_panel_base.parquet
# 3) 如果你已经手动下载/放在某个目录，也可以在 MANUAL_* 里写死路径（最省事）

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_ROOT = os.path.join(PROJECT_ROOT, "Data")

# ====== 如果你想手动指定，填这里（否则留空字符串）======
MANUAL_SENT_PATH = ""  # e.g. r"D:\...\sentiment5_top_level.parquet"
MANUAL_THREAD_PANEL_PATH = ""  # e.g. r"D:\...\thread_panel_base.parquet"

def pick_latest_dir(pattern: str) -> str:
    candidates = sorted(glob.glob(os.path.join(DATA_ROOT, pattern)), reverse=True)
    return candidates[0] if candidates else ""

def find_file_in_dir(base_dir: str, filename: str) -> str:
    if not base_dir:
        return ""
    p = os.path.join(base_dir, filename)
    return p if os.path.exists(p) else ""

# 1) sentiment5_top_level.parquet
if MANUAL_SENT_PATH and os.path.exists(MANUAL_SENT_PATH):
    SENT_PATH = MANUAL_SENT_PATH
else:
    latest_04 = pick_latest_dir("data_04_sentiment5_*")
    SENT_PATH = find_file_in_dir(latest_04, "sentiment5_top_level.parquet")

if not SENT_PATH or not os.path.exists(SENT_PATH):
    raise FileNotFoundError(
        "❌ 找不到 sentiment5_top_level.parquet。\n"
        "请确认你已经跑完 04，并在 Data/data_04_sentiment5_*/ 下存在 sentiment5_top_level.parquet，\n"
        "或者在 MANUAL_SENT_PATH 指定文件绝对路径。"
    )

# 2) thread_panel_base.parquet
if MANUAL_THREAD_PANEL_PATH and os.path.exists(MANUAL_THREAD_PANEL_PATH):
    THREAD_PANEL_PATH = MANUAL_THREAD_PANEL_PATH
else:
    # 你现在 02 的输出目录命名可能不同，我这里用宽松匹配：
    # 如果你固定输出到 Data/data_02_*，并把 thread_panel_base.parquet 放在根目录/子目录都可以加候选
    latest_02 = pick_latest_dir("data_02_*")
    THREAD_PANEL_PATH = ""
    if latest_02:
        # 常见位置：根目录 或 comments/ 或 outputs/
        for sub in ["", "comments", "outputs"]:
            cand = os.path.join(latest_02, sub, "thread_panel_base.parquet")
            if os.path.exists(cand):
                THREAD_PANEL_PATH = cand
                break

# 如果还是找不到，就尝试当前工作目录（你可能把 parquet 直接放在 notebook 同级）
if not THREAD_PANEL_PATH:
    cand = os.path.join(os.getcwd(), "thread_panel_base.parquet")
    if os.path.exists(cand):
        THREAD_PANEL_PATH = cand

if not THREAD_PANEL_PATH or not os.path.exists(THREAD_PANEL_PATH):
    raise FileNotFoundError(
        "❌ 找不到 thread_panel_base.parquet。\n"
        "请确认你 02 阶段输出了 thread_panel_base.parquet，并放在 Data/data_02_*/ 下，\n"
        "或者在 MANUAL_THREAD_PANEL_PATH 指定绝对路径。"
    )

print("✅ SENT_PATH:", SENT_PATH)
print("✅ THREAD_PANEL_PATH:", THREAD_PANEL_PATH)


In [ ]:
# 📌 Cell 3 - Create output folder (single folder per run)
# -------------------------------------------------------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(DATA_ROOT, f"data_05_scorecard_{timestamp}")
os.makedirs(RUN_DIR, exist_ok=True)

OUT_VIDEO_METRICS = os.path.join(RUN_DIR, "video_metrics.csv")
OUT_BOOTSTRAP = os.path.join(RUN_DIR, "bootstrap_results.csv")
OUT_SCORECARD_VIDEO = os.path.join(RUN_DIR, "scorecard_video.csv")
OUT_SCORECARD_CELL = os.path.join(RUN_DIR, "scorecard_cell_summary.csv")
OUT_REPORT = os.path.join(RUN_DIR, "report.json")


print("✅ RUN_DIR:", RUN_DIR)


In [ ]:
# 📌 Cell 4 - Load inputs
# ----------------------
sent = pd.read_parquet(SENT_PATH)
tpb = pd.read_parquet(THREAD_PANEL_PATH)

print("✅ sentiment5_top_level shape:", sent.shape)
print("✅ thread_panel_base shape:", tpb.shape)

sent.head(2)


In [ ]:
# 📌 Cell 5 - Basic schema checks
# ------------------------------
# 05 主线最少需要：
# - sentiment：thread_id, video_id, cell_id, sent5_label
# - thread_panel_base：thread_id, video_id, cell_id, engagement_index, replies_fetched_count, like_count_parent, like_reply_sum
#
# 如果缺字段，先 print 出来再告诉我，我再帮你对齐。

SENT_REQ = ["thread_id", "video_id", "cell_id", "sent5_label"]
TPB_REQ = ["thread_id", "video_id", "cell_id", "engagement_index", "replies_fetched_count", "like_count_parent", "like_reply_sum"]

missing_sent = [c for c in SENT_REQ if c not in sent.columns]
missing_tpb = [c for c in TPB_REQ if c not in tpb.columns]

if missing_sent:
    raise ValueError(f"❌ sentiment5_top_level missing columns: {missing_sent}")
if missing_tpb:
    raise ValueError(f"❌ thread_panel_base missing columns: {missing_tpb}")

# 统一类型
for df in [sent, tpb]:
    df["thread_id"] = df["thread_id"].astype(str)
    df["video_id"] = df["video_id"].astype(str)
    df["cell_id"] = df["cell_id"].astype(str)

# 数值列
tpb["engagement_index"] = pd.to_numeric(tpb["engagement_index"], errors="coerce").fillna(0.0)
tpb["replies_fetched_count"] = pd.to_numeric(tpb["replies_fetched_count"], errors="coerce").fillna(0).astype(int)
tpb["like_count_parent"] = pd.to_numeric(tpb["like_count_parent"], errors="coerce").fillna(0).astype(int)
tpb["like_reply_sum"] = pd.to_numeric(tpb["like_reply_sum"], errors="coerce").fillna(0).astype(int)

print("✅ Schema check passed.")


In [ ]:
# 📌 Cell 6 - Build thread-level merged table
# ------------------------------------------
# 合并原则：
# - 以 thread_id 为主键
# - thread_panel_base 提供互动结构
# - sentiment 提供情绪标签
#
# 如果两边都有 brand/topic 等字段，这里优先保留 thread_panel_base（更像结构表）

common_cols = ["thread_id", "video_id", "cell_id"]

# 只取 sentiment 的必要字段，避免重复列名太多
sent_keep = common_cols + ["sent5_label"]
if "sent5_valence_expected" in sent.columns:
    sent_keep += ["sent5_valence_expected"]
if "sent5_extremity" in sent.columns:
    sent_keep += ["sent5_extremity"]
if "sent5_uncertainty_entropy" in sent.columns:
    sent_keep += ["sent5_uncertainty_entropy"]

sent_small = sent[sent_keep].copy()

# 合并
thread_df = tpb.merge(sent_small, on=common_cols, how="inner")

print("✅ thread_df shape:", thread_df.shape)
thread_df.head(2)


In [ ]:
# 📌 Cell 7 - Compute video-level metrics (from thread_df)
# -------------------------------------------------------
# 你 05 计分卡的 4 个主指标：
# 1) negative_share = % (negative + strongly negative)
# 2) net_negativity = %neg - %pos
# 3) strong_negative_share = % strongly negative
# 4) engagement_index_video = mean(thread_engagement_index)  (你要求用均值表示整体)
#
# 同时输出一些辅助字段：
# - n_threads_used
# - replies_sum / likes_sum
# - engagement_sum（可用于 dashboard，但不进 scorecard）

NEG_SET = set(["negative", "strongly negative"])
POS_SET = set(["positive", "strongly positive"])
STRONG_NEG = "strongly negative"

thread_df["is_neg"] = thread_df["sent5_label"].isin(NEG_SET).astype(int)
thread_df["is_pos"] = thread_df["sent5_label"].isin(POS_SET).astype(int)
thread_df["is_strong_neg"] = (thread_df["sent5_label"] == STRONG_NEG).astype(int)

video_metrics = (
    thread_df.groupby(["video_id", "cell_id"])
    .agg(
        n_threads_used=("thread_id", "count"),
        negative_share=("is_neg", "mean"),
        pos_share=("is_pos", "mean"),
        strong_negative_share=("is_strong_neg", "mean"),

        # engagement: mean (scorecard uses mean)
        engagement_index_mean=("engagement_index", "mean"),

        # extra: sums for dashboard
        engagement_index_sum=("engagement_index", "sum"),
        replies_sum=("replies_fetched_count", "sum"),
        like_parent_sum=("like_count_parent", "sum"),
        like_reply_sum=("like_reply_sum", "sum"),
    )
    .reset_index()
)

video_metrics["net_negativity"] = video_metrics["negative_share"] - video_metrics["pos_share"]

# Weight for weighted bootstrap (and optional weighted means)
video_metrics["w_threads"] = np.log1p(video_metrics["n_threads_used"].astype(float))

print("✅ video_metrics rows:", len(video_metrics))
video_metrics.head(5)
# ✅ 辅助：讨论结构指标（不进入 scorecard，供 06/07 展示）
video_metrics["replies_per_thread"] = video_metrics["replies_sum"] / (video_metrics["n_threads_used"].replace(0, np.nan))
video_metrics["replies_per_thread"] = video_metrics["replies_per_thread"].fillna(0.0)



In [ ]:
# 📌 Cell 8 - Weighted video-block bootstrap
# ----------------------------------------
# 你现在的主线：用 video 为抽样单位（block），并用权重处理“每视频 threads 不均匀”
#
# 权重默认：w = log(1 + n_threads_used)
# 组统计量：weighted mean
#
# 输出：
# - delta_hat（原始样本的 A-B 差）
# - CI_95（bootstrap 分位数）
# - P(delta>0)（方向概率）

def weighted_mean(x: np.ndarray, w: np.ndarray, eps: float = 1e-12) -> float:
    w = np.asarray(w, dtype=float)
    x = np.asarray(x, dtype=float)
    denom = w.sum()
    if denom <= eps:
        return float(np.mean(x)) if len(x) else float("nan")
    return float((w * x).sum() / denom)

def bootstrap_delta_weighted(
    dfA: pd.DataFrame,
    dfB: pd.DataFrame,
    metric: str,
    wcol: str = "w_threads",
    B: int = 1000,
    seed: int = 42
) -> Dict[str, Any]:
    rng = np.random.default_rng(seed)
    xA = dfA[metric].to_numpy()
    wA = dfA[wcol].to_numpy()
    xB = dfB[metric].to_numpy()
    wB = dfB[wcol].to_numpy()

    # point estimate on original sample
    meanA = weighted_mean(xA, wA)
    meanB = weighted_mean(xB, wB)
    delta_hat = meanA - meanB

    deltas = np.empty(B, dtype=float)
    nA, nB = len(dfA), len(dfB)

    for b in range(B):
        idxA = rng.integers(0, nA, size=nA)  # resample videos with replacement
        idxB = rng.integers(0, nB, size=nB)

        deltas[b] = weighted_mean(xA[idxA], wA[idxA]) - weighted_mean(xB[idxB], wB[idxB])

    ci_low, ci_high = np.percentile(deltas, [2.5, 97.5])
    prob_gt0 = float((deltas > 0).mean())

    return {
        "metric": metric,
        "delta_hat": float(delta_hat),
        "ci_low": float(ci_low),
        "ci_high": float(ci_high),
        "prob_delta_gt0": prob_gt0,
        "B": B,
        "wcol": wcol
    }

# Define contrasts (business-friendly)
CONTRASTS = [
    ("shein_labor", "non_shein_labor"),
    ("shein_labor", "shein_environment"),
    ("shein_labor", "non_shein_environment"),
]

METRICS = [
    "negative_share",
    "net_negativity",
    "engagement_index_mean",
]

bootstrap_rows = []
B = 10000  # you can raise to 2000 later if needed

for (A, Bcell) in CONTRASTS:
    dfA = video_metrics[video_metrics["cell_id"] == A].copy()
    dfB = video_metrics[video_metrics["cell_id"] == Bcell].copy()

    if len(dfA) < 3 or len(dfB) < 3:
        print(f"⚠️ Skip contrast {A} vs {Bcell}: too few videos (A={len(dfA)}, B={len(dfB)})")
        continue

    for m in METRICS:
        res = bootstrap_delta_weighted(dfA, dfB, metric=m, wcol="w_threads", B=B, seed=42)
        res["groupA"] = A
        res["groupB"] = Bcell
        res["n_videos_A"] = int(len(dfA))
        res["n_videos_B"] = int(len(dfB))
        bootstrap_rows.append(res)

bootstrap_df = pd.DataFrame(bootstrap_rows)
bootstrap_df


In [ ]:
# 📌 Cell 9 - Min-max scorecard (video-level)
# ------------------------------------------
# 你已决定：min-max 足够 + cell 用均值衡量整体
#
# 四个指标（风险方向都是“越大越风险”）：
# - negative_share
# - net_negativity
# - strong_negative_share
# - engagement_index_mean

SCORECARD_COMPONENTS = [
    "negative_share",
    "net_negativity",
    "engagement_index_mean"
]

WEIGHTS = {
    "negative_share": 0.45,
    "net_negativity": 0.35,
    "engagement_index_mean": 0.20
}

def minmax_series(s: pd.Series) -> pd.Series:
    s = s.astype(float)
    mn, mx = s.min(), s.max()
    if mx - mn < 1e-12:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mn) / (mx - mn)

score_df = video_metrics.copy()

# min-max normalize each component globally (all videos together)
for c in SCORECARD_COMPONENTS:
    score_df[c + "_mm"] = minmax_series(score_df[c])

# weighted sum
score_df["risk_score"] = 0.0
for c, w in WEIGHTS.items():
    score_df["risk_score"] += w * score_df[c + "_mm"]

score_df.head(5)


In [ ]:
# 📌 Cell 10 - Cell-level summary
# ------------------------------
cell_summary = (
    score_df.groupby("cell_id")
    .agg(
        n_videos=("video_id", "count"),
        mean_risk=("risk_score", "mean"),
        std_risk=("risk_score", "std"),
        mean_negative_share=("negative_share", "mean"),
        mean_net_negativity=("net_negativity", "mean"),
        mean_strong_neg_share=("strong_negative_share", "mean"),
        mean_engagement=("engagement_index_mean", "mean"),
        mean_threads=("n_threads_used", "mean"),
        total_threads=("n_threads_used", "sum"),
        total_replies=("replies_sum", "sum"),
        mean_replies_per_thread=("replies_sum", lambda s: 0),  # 先占位，下面我们用更稳的方式算
        median_risk=("risk_score", "median"),

    )
    .reset_index()
    .sort_values("mean_risk", ascending=False)
)
# 用视频层 mean(replies_per_thread) 更合理（避免总量被视频数影响）
tmp = score_df.groupby("cell_id")["replies_per_thread"].mean().reset_index(name="mean_replies_per_thread")
cell_summary = cell_summary.merge(tmp, on="cell_id", how="left")
cell_summary


In [ ]:
# 📌 Cell 11 - Save outputs
# ------------------------
video_metrics.to_csv(OUT_VIDEO_METRICS, index=False, encoding="utf-8-sig")
bootstrap_df.to_csv(OUT_BOOTSTRAP, index=False, encoding="utf-8-sig")
score_df.to_csv(OUT_SCORECARD_VIDEO, index=False, encoding="utf-8-sig")
cell_summary.to_csv(OUT_SCORECARD_CELL, index=False, encoding="utf-8-sig")

print("✅ Saved:", OUT_VIDEO_METRICS)
print("✅ Saved:", OUT_BOOTSTRAP)
print("✅ Saved:", OUT_SCORECARD_VIDEO)
print("✅ Saved:", OUT_SCORECARD_CELL)


In [ ]:
# 📌 Cell 12 - Save report.json
# ----------------------------
report: Dict[str, Any] = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "run_dir": RUN_DIR,
    "inputs": {
        "sentiment5_top_level": SENT_PATH,
        "thread_panel_base": THREAD_PANEL_PATH
    },
    "design": {
        "contrasts": [{"groupA": a, "groupB": b} for a, b in CONTRASTS],
        "metrics_for_bootstrap": METRICS,
        "bootstrap_B": int(B),
        "weight_scheme": "w_threads = log1p(n_threads_used)",
        "scorecard_components": SCORECARD_COMPONENTS,
        "scorecard_weights": WEIGHTS,
        "cell_score_aggregation": "mean(video_risk_score)"
    },
    "outputs": {
        "video_metrics_csv": OUT_VIDEO_METRICS,
        "bootstrap_results_csv": OUT_BOOTSTRAP,
        "scorecard_video_csv": OUT_SCORECARD_VIDEO,
        "scorecard_cell_summary_csv": OUT_SCORECARD_CELL,
    },
    "quick_summary": {
        "n_videos_total": int(len(score_df)),
        "cells": cell_summary.to_dict(orient="records")
    }
}

with open(OUT_REPORT, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("✅ Saved:", OUT_REPORT)


In [ ]:
# 📌 Cell 13 - Quality Gate print
# ------------------------------
print("\n" + "="*50)
print("🚦 QUALITY GATE PASSED: 05 SCORECARD ASSETS DELIVERED")
print("="*50)
print("📌 Output folder:", RUN_DIR)
print("\n📊 Cell mean risk (higher = riskier):")
print(cell_summary[["cell_id", "n_videos", "mean_risk", "std_risk", "total_threads", "total_replies"]].to_string(index=False))

print("\n📌 Key contrasts (bootstrap) preview:")
print(bootstrap_df[["groupA", "groupB", "metric", "delta_hat", "ci_low", "ci_high", "prob_delta_gt0"]].head(12).to_string(index=False))
